# ⚡ Train Dora-X2 16-Layer MH-RTU Chat Model on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Baba01hacker666/doranerual/blob/main/notebooks/train_dora_x2_colab.ipynb)

**Dora-X2 Specifications:**
- **Architecture**: 16 Layers | 768 Dimension | 12 Recurrent Heads (head_dim=64)
- **Sequence Engine**: Multi-Head Recurrent Trace Units (MH-RTU) with learned channel decay
- **Inference Memory**: Exact (1)$ Constant State Memory (<50 KB total across all 16 layers)
- **FeedForward**: Standard SwiGLU MLP (~123.1M Parameters)
- **Tokenization**: Zero-dependency raw UTF-8 byte stream (Vocab size = 256)


## 1. Clone Repository & Install Dependencies


In [ ]:
!git clone https://github.com/Baba01hacker666/doranerual.git
%cd doranerual
!pip install -r requirements.txt


## 2. Verify Architecture & Build Native C++ SIMD Engine


In [ ]:
import doraneural as dn
from doraneural.dora_x2 import DoraX2Config, DoraX2LM

print("CPU Architecture:", dn.get_cpu_arch())
print("Building Native C++ Engine...")
so_file = dn.build_cpp_library(force=True)
print("Compiled C++ Library:", so_file)

cfg = DoraX2Config()
print(f"Dora-X2 Config: {cfg.n_layers} Layers, {cfg.dim} Dim, {cfg.n_heads} Heads")
print(f"Total Parameters: {cfg.parameter_count:,} ({cfg.parameter_count/1e6:.1f}M)")


## 3. Train Dora-X2 (Cloud Execution)
Run sequence chunk training with AdamW on Google Colab hardware.


In [ ]:
!python research/train_dora_x2.py     --output-dir checkpoints/dora_x2     --dim 768     --layers 16     --heads 12     --epochs 3     --max-bytes 100000     --lr 0.001     --tag colab_dora_x2_run


## 4. Interactive Conversational Chat Session
Stream responses in real time using the (1)$ constant state engine.


In [ ]:
from doraneural.dora_x2 import DoraX2LM, DoraX2ChatSession

checkpoint_path = "checkpoints/dora_x2/colab_dora_x2_run"
print(f"Loading checkpoint: {checkpoint_path}")
model = DoraX2LM.load(checkpoint_path)
session = DoraX2ChatSession(model)

# Multi-turn streaming chat probe
query = "Hello Dora-X2! How does your 16-layer MH-RTU architecture achieve O(1) state memory?"
print(f"User: {query}\n")
print("Assistant: ", end="", flush=True)
for chunk in session.chat(query, max_tokens=100, stream=True):
    print(chunk, end="", flush=True)
print("\n")


## 5. Save Checkpoints to Google Drive (Optional)


In [ ]:
# Uncomment to mount Google Drive and save model weights permanently
# from google.colab import drive
# import shutil
# from pathlib import Path
# drive.mount("/content/drive")
# dest = Path("/content/drive/MyDrive/dora_x2_checkpoints")
# dest.mkdir(parents=True, exist_ok=True)
# shutil.copytree("checkpoints/dora_x2", dest / "colab_latest", dirs_exist_ok=True)
# print(f"Checkpoint persisted to {dest / 'colab_latest'}")
